In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("15-broadcast-skew-joins")
dfs = register_views(spark)
emp  = dfs["employees"]
dept = dfs["departments"]
sal  = dfs["salary_history"]
pa   = dfs["project_assignments"]

# ── Section 1: What broadcast join is ────────────────────────────────────────
# Small table is serialised and sent to every executor; large table stays in place.
# No shuffle on either side → avoids Exchange entirely.
# Default threshold: 10 MB (autoBroadcastJoinThreshold); tables below this are auto-broadcast.
print("\n=== Section 1: Broadcast join threshold ===")
print("Auto broadcast threshold (bytes):", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
# dept has 8 rows; emp has 41 rows — both fit well under 10 MB in local mode
# UI: SQL tab will show BroadcastHashJoin, no Exchange nodes

# ── Section 2: Explicit broadcast hint ───────────────────────────────────────
# broadcast() wraps a DataFrame to force BroadcastHashJoin regardless of table size
# Useful when the planner mis-estimates size and picks SortMergeJoin
print("\n=== Section 2: Explicit broadcast hint ===")
from pyspark.sql.functions import broadcast
result = emp.join(broadcast(dept), "dept_id", "left")
result.select("emp_id", "first_name", "dept_name").show(5)

emp.join(broadcast(dept), "dept_id", "left").explain()
# Look for: BroadcastHashJoin in plan — no Exchange on the dept side

# ── Section 3: Disable auto broadcast to see SortMergeJoin ───────────────────
# Setting threshold to -1 disables auto broadcast → Spark falls back to SortMergeJoin
# SortMergeJoin requires both sides to be sorted on the join key → two Exchange nodes
print("\n=== Section 3: SortMergeJoin (auto broadcast disabled) ===")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
emp.join(dept, "dept_id", "left").explain()
# Look for: Exchange + Sort on both sides, then SortMergeJoin
# Re-enable:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

# ── Section 4: Demonstrating data skew ───────────────────────────────────────
# Engineering (dept_id=1) has 14/41 employees = 34% of rows → one shuffle partition
# receives disproportionately more rows when joining on emp_id that maps to dept_id=1
print("\n=== Section 4: Data skew partition distribution ===")
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # force shuffle join
joined = emp.join(sal, "emp_id", "inner")
joined.withColumn("pid", F.spark_partition_id()) \
      .groupBy("pid").count() \
      .orderBy("pid").show()
# Some partitions will have more rows — that is skew
# UI: Stages → task duration bars; the stragglers are the skewed partitions
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

# ── Section 5: Salting to fix skew (manual approach) ─────────────────────────
# Salting breaks a hot key into NUM_SALT virtual keys, spreading rows across partitions.
# Large side: assign each row a random salt [0, NUM_SALT).
# Small side: explode each row NUM_SALT times, one per salt value.
# Join on (original_key, salt) → hot-key rows are spread across NUM_SALT partitions.
print("\n=== Section 5: Salting to fix skew ===")
import random
NUM_SALT = 4

# Replicate the small side (dept) once per salt value
salt_dept = dept.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(NUM_SALT)])))

# Assign a deterministic salt to the large side (emp) based on emp_id
# Using modulo keeps results reproducible; random would be equally valid
emp_salted = emp.withColumn("salt", (F.col("emp_id") % NUM_SALT).cast(IntegerType()))

# Join on (dept_id, salt) — Engineering rows are now spread across 4 virtual keys
result = (
    emp_salted
    .join(
        salt_dept,
        (emp_salted.dept_id == salt_dept.dept_id) & (emp_salted.salt == salt_dept.salt),
        "left"
    )
    .select(emp_salted.emp_id, emp_salted.first_name, salt_dept.dept_name)
)
result.show(5)
# Note: data is tiny here; salting pays off in production when one key has millions of rows
# UI: partition sizes become more balanced after salting

# ── Section 6: AQE skew join handling (automatic) ────────────────────────────
# AQE detects skewed partitions AT RUNTIME and splits them automatically.
# No code change required — enable the two configs and Spark handles the rest.
print("\n=== Section 6: AQE skew join ===")
spark.conf.set("spark.sql.adaptive.enabled",          "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
emp.join(sal, "emp_id", "inner") \
   .groupBy("dept_id").agg(F.sum("salary_after").alias("total_salary_after")) \
   .show()
# UI: SQL tab → AQE plan shows "CustomShuffleReader" replacing Exchange when skew detected
# In real production workloads (millions of rows), AQE splits the skewed partition
# into smaller sub-tasks automatically — no manual salting needed

# ── Section 7: BroadcastNestedLoopJoin (non-equi join) ───────────────────────
# Non-equi joins (>, <, !=, BETWEEN) cannot use hash-based join strategies.
# Spark falls back to BroadcastNestedLoopJoin: O(M*N) nested iteration, small side broadcast.
print("\n=== Section 7: BroadcastNestedLoopJoin (non-equi) ===")
emp.join(dept, emp.salary > dept.budget, "inner") \
   .select(emp.emp_id, emp.salary, dept.dept_name, dept.budget) \
   .show(5)
emp.join(dept, emp.salary > dept.budget, "inner").explain()
# Look for: BroadcastNestedLoopJoin in plan (not BroadcastHashJoin or SortMergeJoin)
# Note: emp 10 & 15 have salary=NULL → NULL comparisons evaluate to false, those rows excluded
# UI: single stage; no Exchange because dept is broadcast; but task time is O(M*N)

stop_and_wait(spark)